# Atividade: tratamento de uma base de clientes

Complete as células com código pandas. A base tratada deve ficar adequada para análise e aprendizado de máquina.

In [4]:
# Importe a biblioteca pandas.
import pandas as pd
import numpy as np

In [5]:
# Carregue o arquivo clientes_sem_tratamento.csv em um DataFrame.
df = pd.read_csv('clientes_sem_tratamento.csv')

In [6]:
# Inspecione as primeiras e as últimas linhas, as dimensões, os nomes das colunas e os tipos de dados.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_cliente     22 non-null     int64
 1   nome           22 non-null     str  
 2   email          21 non-null     str  
 3   idade          21 non-null     str  
 4   cidade         21 non-null     str  
 5   estado         22 non-null     str  
 6   renda_mensal   21 non-null     str  
 7   data_cadastro  22 non-null     str  
 8   ativo          22 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.7 KB


In [7]:
# Conte os valores ausentes de cada coluna e procure strings que também representem ausência, como 'não informado'.
print("valores ausentes")
print(df.isnull().sum())

valores ausentes
id_cliente       0
nome             0
email            1
idade            1
cidade           1
estado           0
renda_mensal     1
data_cadastro    0
ativo            0
dtype: int64


In [8]:
# Identifique registros totalmente duplicados e IDs repetidos. Remova a cópia repetida do cliente de id 2.
print(df.shape)
df = df.drop_duplicates(subset='id_cliente', keep='first')
print(df.shape)

(22, 9)
(21, 9)


In [9]:
# Remova espaços extras no início e no fim das colunas textuais.
colunas_texto = df.select_dtypes(include='object').columns
for col in colunas_texto:
    print(col)
    df[col] = df[col].str.strip()


nome
email
idade
cidade
estado
renda_mensal
data_cadastro
ativo


/tmp/ipykernel_21177/2118166119.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df.select_dtypes(include='object').columns


In [10]:
print("nomes e emails sem padronização")
df[['nome', 'email']]

nomes e emails sem padronização


,nome,email
0,Ana Paula,ANA.PAULA@email.com
1,Bruno Lima,bruno.lima@email.com
2,Carla Souza,NaN
3,Diego Alves,diego.alves@email.com
4,Elisa Rocha,ELISA.ROCHA@EMAIL.COM
5,Fábio Mendes,fabio.mendes@email.com
6,Gabriela Nunes,gabriela.nunes@email.com
7,Hugo Martins,hugo.martins@email
9,Ana Paula,ana.paula@email.com
10,Igor Freitas,igor.freitas@email.com


In [12]:
# Padronize os nomes de pessoas com iniciais maiúsculas e os e-mails com letras minúsculas.
if 'nome' in df.columns:
    df['nome'] = df['nome'].str.title()
if 'email' in df.columns:
    df['email'] = df['email'].str.lower()

In [13]:
print("nomes e emails padronizados")
df[['nome', 'email']]

nomes e emails padronizados


,nome,email
0,Ana Paula,ana.paula@email.com
1,Bruno Lima,bruno.lima@email.com
2,Carla Souza,NaN
3,Diego Alves,diego.alves@email.com
4,Elisa Rocha,elisa.rocha@email.com
5,Fábio Mendes,fabio.mendes@email.com
6,Gabriela Nunes,gabriela.nunes@email.com
7,Hugo Martins,hugo.martins@email
9,Ana Paula,ana.paula@email.com
10,Igor Freitas,igor.freitas@email.com


In [14]:
# Localize e remova e-mails inválidos e cadastros duplicados pelo mesmo e-mail. Preserve o valor ausente do e-mail do cliente 3.
import re

# Padrão simples de validação de email
email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

# Marcar emails inválidos (exceto NaN)
df['email_valido'] = df['email'].apply(lambda x: bool(re.match(email_pattern, str(x))) if pd.notna(x) else True)
print(f"Emails inválidos encontrados: {(~df['email_valido']).sum()}")

# Remover emails inválidos (mas não remover cliente 3 se email for NaN)
df = df[df['email_valido'] | (df['id_cliente'] == 3)]
df = df.drop('email_valido', axis=1)

# Remover duplicatas de email (preservando NaN do cliente 3)
df_sem_na = df[df['email'].notna()]
df_com_na = df[df['email'].isna()]
df_sem_na = df_sem_na.drop_duplicates(subset='email', keep='first')
df = pd.concat([df_sem_na, df_com_na]).sort_index()
print(f"Dimensões após limpeza de emails: {df.shape}")

Emails inválidos encontrados: 1
Dimensões após limpeza de emails: (19, 9)


In [15]:
# Converta idade para número. Remova idades impossíveis e preencha a idade ausente com a mediana das idades válidas.

df['idade'] = pd.to_numeric(df['idade'], errors='coerce')

# Remover idades impossíveis
df.loc[(df['idade'] < 0) | (df['idade'] > 120), 'idade'] = np.nan

# Preencher ausentes com mediana
mediana_idade = df['idade'].median()
df['idade'] = df['idade'].fillna(mediana_idade)

print(f"Mediana de idade: {mediana_idade}")
print(f"Idades ausentes preenchidas")

Mediana de idade: 34.0
Idades ausentes preenchidas


In [16]:
# Limpe renda_mensal: remova 'R$', pontos de milhar e espaços; troque a vírgula decimal por ponto; converta para número.

if df['renda_mensal'].dtype == 'object':
    df['renda_mensal'] = df['renda_mensal'].str.replace('R$', '', regex=False)
    df['renda_mensal'] = df['renda_mensal'].str.replace('.', '', regex=False)
    df['renda_mensal'] = df['renda_mensal'].str.replace(' ', '', regex=False)
    df['renda_mensal'] = df['renda_mensal'].str.replace(',', '.', regex=False)
df['renda_mensal'] = pd.to_numeric(df['renda_mensal'], errors='coerce')
print("renda_mensal convertida para número")

renda_mensal convertida para número


In [17]:
# Considere rendas negativas e 'não informado' como ausentes. Preencha rendas ausentes com a mediana das rendas válidas.
df.loc[df['renda_mensal'] < 0, 'renda_mensal'] = np.nan
mediana_renda = df['renda_mensal'].median()
df['renda_mensal'] = df['renda_mensal'].fillna(mediana_renda)
print(f"Mediana da renda: {mediana_renda}")

Mediana da renda: 4200.0


In [18]:
# Converta data_cadastro para data, aceitando os formatos existentes. Remova registros cuja data seja impossível.
df['data_cadastro'] = pd.to_datetime(df['data_cadastro'], errors='coerce', dayfirst=True)
# Remover registros com datas inválidas
qtd_antes = len(df)
df = df[df['data_cadastro'].notna()]
print(f"Registros removidos por data inválida: {qtd_antes - len(df)}")
print(f"Dimensões após limpeza de datas: {df.shape}")
df

Registros removidos por data inválida: 5
Dimensões após limpeza de datas: (14, 9)


,id_cliente,nome,email,idade,cidade,estado,renda_mensal,data_cadastro,ativo
0,1,Ana Paula,ana.paula@email.com,22.0,sao paulo,sp,4200.0,2026-02-01,Sim
3,4,Diego Alves,diego.alves@email.com,34.0,Sorocaba,SP,4200.0,2026-02-10,Sim
5,6,Fábio Mendes,fabio.mendes@email.com,27.0,campinas,SP,4200.0,2026-02-13,Não
6,7,Gabriela Nunes,gabriela.nunes@email.com,33.0,São Paulo,sp,4200.0,2026-02-14,S
10,11,Igor Freitas,igor.freitas@email.com,34.0,Guarulhos,SP,4200.0,2026-02-18,Sim
11,12,Juliana Costa,juliana.costa@email.com,31.0,NaN,SP,4200.0,2026-02-19,N
12,13,Kleber Dias,kleber.dias@email.com,45.0,Osasco,sp,4200.0,2026-02-20,não
14,15,Marcos Silva,marcos.silva@email.com,39.0,SAO PAULO,SP,4200.0,2026-02-22,1
15,16,Natália Reis,natalia.reis@email.com,28.0,Santos,SP,4200.0,2026-02-23,0
16,17,Otávio Gomes,otavio.gomes@email.com,34.0,Sorocaba,SP,4200.0,2026-02-24,verdadeiro


In [ ]:
# Padronize cidade: corrija caixa, acentuação e grafias equivalentes. Preencha cidade ausente com 'Não informado'.

In [ ]:
# Padronize estado com duas letras maiúsculas. Investigue e remova o registro incompatível com as cidades paulistas da base.

In [ ]:
# Converta as diferentes representações de ativo para os valores booleanos True e False.

In [ ]:
# Reordene pelo id_cliente, redefina o índice e confira novamente dimensões, tipos, ausências, duplicatas e estatísticas.

In [ ]:
# Compare seu resultado com clientes_tratados.csv e explique qualquer diferença encontrada.

In [ ]:
# Salve o DataFrame final em um novo arquivo CSV, sem gravar o índice. Não sobrescreva as bases fornecidas.